In [2]:
import duckdb
import pandas as pd

# Connect to DuckDB
DB_PATH = "developer_project.duckdb"
con = duckdb.connect(DB_PATH)

# Optional display settings
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)


### Load Raw Tables

In [13]:
# Activity raw
con.execute("""
CREATE OR REPLACE TABLE activity_raw AS
SELECT *
FROM read_csv_auto(
    'Data/DEVPM_8674_Cal_Poly_Export_dev_activity.csv',
    delim='|',
    header=True,
    all_varchar=True,
    sample_size=-1
)
""")

# Contact raw
con.execute("""
CREATE OR REPLACE TABLE contact_raw AS
SELECT *
FROM read_csv_auto(
    'Data/DEVPM_8674_Cal_Poly_Export_dev_contact.csv',
    delim=',',
    header=True,
    all_varchar=True,
    sample_size=-1
)
""")

# SDK raw
con.execute("""
CREATE OR REPLACE TABLE sdk_download_raw AS
SELECT *
FROM read_csv_auto(
    'Data/DEVPM_8674_Cal_Poly_Export_sdk_download.csv',
    delim=',',
    header=True,
    all_varchar=True,
    sample_size=-1
)
""")

# Supplement raw
con.execute("""
CREATE OR REPLACE TABLE contact_supplement_raw AS
SELECT *
FROM read_csv_auto(
    'Data/DEVPM_8674_Cal_Poly_Export_dev_contact_supplement.csv',
    delim=',',
    header=True,
    all_varchar=True,
    sample_size=-1,
    ignore_errors=True
)
""")


100% ▕██████████████████████████████████████▏ (00:00:31.42 elapsed)     
100% ▕██████████████████████████████████████▏ (00:00:13.80 elapsed)     
100% ▕██████████████████████████████████████▏ (00:00:19.69 elapsed)     


In [14]:

con.execute("""
SELECT column_name
FROM information_schema.columns
WHERE table_name = 'activity_raw'
ORDER BY ordinal_position
""").fetchdf()


,column_name
0,activity
1,activity_date
2,activity_name
3,activity_type
4,activity_role
5,activity_attendance
6,dev_contact
7,activity_score
8,filepath
9,activity_id


In [15]:

con.execute("""
SELECT column_name
FROM information_schema.columns
WHERE table_name = 'contact_raw'
ORDER BY ordinal_position
""").fetchdf()


,column_name
0,developer_id
1,program_application_source
2,country
3,region
4,sub_region
5,zone
6,territory
7,organization_english_name
8,development_areas
9,other_development_areas


In [16]:

con.execute("""
SELECT column_name
FROM information_schema.columns
WHERE table_name = 'sdk_download_raw'
ORDER BY ordinal_position
""").fetchdf()


,column_name
0,source
1,sdk_name
2,PRODUCTNAME
3,PRODUCTRELEASE
4,country
5,region
6,subregion
7,territory
8,zone
9,downloaddate


In [17]:

con.execute("""
SELECT column_name
FROM information_schema.columns
WHERE table_name = 'contact_supplement_raw'
ORDER BY ordinal_position
""").fetchdf()


,column_name
0,developer_id
1,program_application_source
2,country
3,region
4,sub_region
5,zone
6,territory
7,organization_english_name
8,development_areas
9,other_development_areas


### 2. Create cleaned analysis tables


In [18]:
# Activity clean
con.execute("""
CREATE OR REPLACE TABLE activity_clean AS
SELECT
    dev_contact,
    activity,
    activity_name,
    activity_type,
    activity_role,
    activity_attendance,
    TRY_CAST(activity_score AS DOUBLE) AS activity_score,
    TRY_CAST(activity_date AS TIMESTAMP) AS activity_date,
    activity_id,
    filepath,
    pk1,
    pk2,
    lead_source,
    nvidia_campaign_id,
    gtc_nvidia_campaign_id,
    lead_source_details
FROM activity_raw
""")

# Contact clean
con.execute("""
CREATE OR REPLACE TABLE contact_clean AS
SELECT
    developer_id,
    program_application_source,
    country,
    region,
    sub_region,
    zone,
    territory,
    organization_english_name,
    development_areas,
    other_development_areas,
    industry_segment_vertical,
    other_industry_segment_vertical,
    sub_industry_segment_vertical,
    fields_of_interest,
    other_fields_of_interest,
    TRY_CAST(first_program_application_date AS TIMESTAMP) AS first_program_application_date,
    account_id,
    account_name,
    TRY_CAST(last_activity_date AS TIMESTAMP) AS last_activity_date,
    TRY_CAST(last_modified_date AS TIMESTAMP) AS last_modified_date,
    TRY_CAST(created_date AS TIMESTAMP) AS created_date,
    wwfo_category,
    wwfo_target_list,
    account_industry_segment,
    account_source,
    account_type,
    TRY_CAST(devzone_last_login_date AS TIMESTAMP) AS devzone_last_login_date,
    organization_website,
    inception_id,
    TRY_CAST(first_activity_date AS TIMESTAMP) AS first_activity_date,
    normalized_account_name,
    TRY_CAST(rdp_exit_date AS TIMESTAMP) AS rdp_exit_date
FROM contact_raw
""")

# SDK clean
con.execute("""
CREATE OR REPLACE TABLE sdk_download_clean AS
SELECT
    source,
    sdk_name,
    PRODUCTNAME AS product_name,
    PRODUCTRELEASE AS product_release,
    country,
    region,
    subregion,
    territory,
    zone,
    TRY_CAST(downloaddate AS DATE) AS download_date,
    FILETYPE AS file_type,
    OPERATINGSYSTEM AS operating_system,
    OS_DISTRIBUTION AS os_distribution,
    ARCHITECTURE AS architecture,
    TRY_CAST(KPI AS DOUBLE) AS kpi,
    TRY_CAST(downloadcount AS BIGINT) AS download_count
FROM sdk_download_raw
""")

# Supplement clean
# Keep developer_id as VARCHAR because it is hashed text in this file
# Cast the timestamp-like columns safely
con.execute("""
CREATE OR REPLACE TABLE contact_supplement_clean AS
SELECT
    developer_id,
    program_application_source,
    country,
    region,
    sub_region,
    zone,
    territory,
    organization_english_name,
    development_areas,
    other_development_areas,
    industry_segment_vertical,
    other_industry_segment_vertical,
    sub_industry_segment_vertical,
    fields_of_interest,
    other_fields_of_interest,
    TRY_CAST(first_program_application_date AS TIMESTAMP) AS first_program_application_date,
    account_id,
    account_name,
    TRY_CAST(last_activity_date AS TIMESTAMP) AS last_activity_date,
    TRY_CAST(last_modified_date AS TIMESTAMP) AS last_modified_date,
    TRY_CAST(created_date AS TIMESTAMP) AS created_date,
    wwfo_category,
    wwfo_target_list,
    account_industry_segment,
    account_source,
    account_type,
    TRY_CAST(devzone_last_login_date AS TIMESTAMP) AS devzone_last_login_date,
    organization_website,
    inception_id,
    TRY_CAST(first_activity_date AS TIMESTAMP) AS first_activity_date,
    normalized_account_name,
    TRY_CAST(rdp_exit_date AS TIMESTAMP) AS rdp_exit_date
FROM contact_supplement_raw
""")


100% ▕██████████████████████████████████████▏ (00:00:12.48 elapsed)     
100% ▕██████████████████████████████████████▏ (00:00:06.66 elapsed)     
100% ▕██████████████████████████████████████▏ (00:00:07.91 elapsed)     


### Sanity Checks

In [19]:
print("Tables in database:")
display(con.execute("SHOW TABLES").fetchdf())

print("Activity raw row count:")
display(con.execute("SELECT COUNT(*) AS row_count FROM activity_raw").fetchdf())

print("Contact raw row count:")
display(con.execute("SELECT COUNT(*) AS row_count FROM contact_raw").fetchdf())

print("SDK raw row count:")
display(con.execute("SELECT COUNT(*) AS row_count FROM sdk_download_raw").fetchdf())

print("Supplement raw row count:")
display(con.execute("SELECT COUNT(*) AS row_count FROM contact_supplement_raw").fetchdf())

print("Activity clean schema:")
display(con.execute("DESCRIBE activity_clean").fetchdf())

print("Contact clean schema:")
display(con.execute("DESCRIBE contact_clean").fetchdf())

print("SDK clean schema:")
display(con.execute("DESCRIBE sdk_download_clean").fetchdf())

print("Supplement clean schema:")
display(con.execute("DESCRIBE contact_supplement_clean").fetchdf())

Tables in database:


,name
0,activity_capped
1,activity_clean
2,activity_final
3,activity_raw
4,activity_sample
5,activity_score_bounds
6,activity_score_mapping_clean
7,activity_score_mapping_raw
8,activity_scored
9,activity_stage


Activity raw row count:


,row_count
0,69347526


Contact raw row count:


...

,column_name,column_type,null,key,default,extra
0,developer_id,VARCHAR,YES,None,None,None
1,program_application_source,VARCHAR,YES,None,None,None
2,country,VARCHAR,YES,None,None,None
3,region,VARCHAR,YES,None,None,None
4,sub_region,VARCHAR,YES,None,None,None
5,zone,VARCHAR,YES,None,None,None
6,territory,VARCHAR,YES,None,None,None
7,organization_english_name,VARCHAR,YES,None,None,None
8,development_areas,VARCHAR,YES,None,None,None
9,other_development_areas,VARCHAR,YES,None,None,None


SDK clean schema:


,column_name,column_type,null,key,default,extra
0,source,VARCHAR,YES,None,None,None
1,sdk_name,VARCHAR,YES,None,None,None
2,product_name,VARCHAR,YES,None,None,None
3,product_release,VARCHAR,YES,None,None,None
4,country,VARCHAR,YES,None,None,None
5,region,VARCHAR,YES,None,None,None
6,subregion,VARCHAR,YES,None,None,None
7,territory,VARCHAR,YES,None,None,None
8,zone,VARCHAR,YES,None,None,None
9,download_date,DATE,YES,None,None,None


Supplement clean schema:


,column_name,column_type,null,key,default,extra
0,developer_id,VARCHAR,YES,None,None,None
1,program_application_source,VARCHAR,YES,None,None,None
2,country,VARCHAR,YES,None,None,None
3,region,VARCHAR,YES,None,None,None
4,sub_region,VARCHAR,YES,None,None,None
5,zone,VARCHAR,YES,None,None,None
6,territory,VARCHAR,YES,None,None,None
7,organization_english_name,VARCHAR,YES,None,None,None
8,development_areas,VARCHAR,YES,None,None,None
9,other_development_areas,VARCHAR,YES,None,None,None


In [20]:
con.execute("""
CREATE OR REPLACE TABLE contact_clean_backup_before_supplement AS
SELECT * FROM contact_clean
""")

100% ▕██████████████████████████████████████▏ (00:00:03.69 elapsed)     


In [3]:
con.execute("""
CREATE OR REPLACE TABLE contact_clean_appended AS
SELECT * FROM contact_clean

UNION ALL

SELECT * FROM contact_supplement_clean
""")

100% ▕██████████████████████████████████████▏ (00:00:02.96 elapsed)     


In [4]:
con.execute("DROP TABLE contact_clean")
con.execute("ALTER TABLE contact_clean_appended RENAME TO contact_clean")

In [5]:
con.close()